# 08 — Bracket Output

Generate a full 2025 NCAA tournament bracket prediction using the trained MOE model,
visualize championship probabilities, region-by-region advancement heatmaps,
expert decomposition, and compare MOE picks against a Log5 baseline.

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from src.bracket.predictor import BracketPredictor, BracketPrediction
from src.bracket.visualizer import (
    print_bracket,
    print_advancement_table,
    print_champion_probabilities,
    print_final_four,
    print_round1_decomposition,
)
from src.bracket.structure import load_bracket
from src.models.moe_ensemble import MOEEnsemble
from src.data.merge import merge_kenpom_with_matchups
from src.data.kaggle_loader import load_kenpom
from src.features.pipeline import build_features_for_split
from src.features.context_features import build_upset_rate_lookup
from src.features.kenpom_deltas import compute_deltas
from src.simulation.mc_engine import MCSimulator, SimulationResult
from src.simulation.log5 import build_log5_win_prob_fn
from src.simulation.matchup_builder import TeamStatsLookup
from src.config import MODELS_DIR, FIRST_YEAR, LAST_YEAR, SKIP_YEARS, EXPERT_TYPES

logging.basicConfig(level=logging.INFO)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 2. Load Model

In [ ]:
merged_df = merge_kenpom_with_matchups()
all_seasons = [y for y in range(FIRST_YEAR, LAST_YEAR + 1) if y not in SKIP_YEARS]
train_seasons = [y for y in all_seasons if y != 2025]

try:
    moe = MOEEnsemble.load(MODELS_DIR / "moe")
    print("Loaded pre-trained MOE")
except (FileNotFoundError, Exception) as e:
    print(f"Training MOE from scratch: {e}")
    train_fs, _ = build_features_for_split(train_seasons, [2025], merged_df)
    moe = MOEEnsemble()
    moe.train_full_nested(train_fs, train_seasons, merged_df)

print(f"Experts: {list(moe.experts.keys())}")
print(f"Gating: {'trained' if moe.gating is not None else 'uniform fallback'}")

## 3. Generate Prediction

In [ ]:
predictor = BracketPredictor(moe, n_simulations=50000, random_seed=42)
prediction = predictor.predict_historical(2025, merged_df=merged_df)

print(f"Season: {prediction.season}")
print(f"Champion: {prediction.champion}")
print(f"Simulations: {prediction.simulation.n_simulations:,}")
print(f"Total picks: {len(prediction.picks)}")

## 4. Championship Probabilities

In [ ]:
print_champion_probabilities(prediction, top_n=16)

# Build data for horizontal bar chart
bracket = prediction.bracket
sim = prediction.simulation

# Map team_id -> (name, seed, region)
team_info = {}
for seed_str, (tid, tname, snum) in bracket.teams.items():
    region = seed_str[0]  # W, X, Y, Z
    if tid not in team_info or snum < team_info[tid][1]:
        team_info[tid] = (tname, snum, region)

# Top 16 by championship probability
sorted_teams = sorted(sim.championship_probs.items(), key=lambda x: -x[1])[:16]

names = []
probs = []
colors = []
region_colors = {"W": "#E24A33", "X": "#348ABD", "Y": "#988ED5", "Z": "#FBC15E"}
region_labels = {"W": "West", "X": "East", "Y": "South", "Z": "Midwest"}

for tid, prob in sorted_teams:
    tname, seed, region = team_info.get(tid, (f"Team{tid}", 0, "?"))
    names.append(f"({seed}) {tname}")
    probs.append(prob * 100)
    colors.append(region_colors.get(region, "#999999"))

fig, ax = plt.subplots(figsize=(10, 7))
y_pos = range(len(names) - 1, -1, -1)
ax.barh(y_pos, probs, color=colors, edgecolor="black", linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.set_xlabel("Championship Probability (%)")
ax.set_title(f"{prediction.season} Championship Probabilities — Top 16")

# Legend for regions
legend_patches = [
    mpatches.Patch(color=c, label=region_labels.get(r, r))
    for r, c in region_colors.items()
]
ax.legend(handles=legend_patches, loc="lower right", title="Region")

plt.tight_layout()
plt.show()

## 5. Final Four

In [ ]:
print_final_four(prediction)

# Visualize Final Four matchups as a simple bracket diagram
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")
ax.set_title(f"{prediction.season} Final Four", fontsize=16, fontweight="bold")

ff = prediction.final_four
champ = prediction.champion

# Semifinal boxes (left side: top semifinal, bottom semifinal)
box_props = dict(boxstyle="round,pad=0.4", facecolor="lightsteelblue", edgecolor="black")
champ_box = dict(boxstyle="round,pad=0.5", facecolor="gold", edgecolor="black", linewidth=2)

# Position Final Four teams
positions = [(1.5, 8), (1.5, 2), (8.5, 8), (8.5, 2)]
for i, team in enumerate(ff[:4]):
    x, y = positions[i] if i < len(positions) else (5, 5)
    label = f"({team['seed']}) {team['team_name']}\n{team['probability']*100:.1f}%"
    ax.text(x, y, label, fontsize=11, ha="center", va="center", bbox=box_props)

# Champion in the center
if champ:
    champ_label = f"CHAMPION\n({champ['seed']}) {champ['team_name']}\n{champ['probability']*100:.1f}%"
    ax.text(5, 5, champ_label, fontsize=13, ha="center", va="center",
            bbox=champ_box, fontweight="bold")

# Draw connecting lines from semis to champion
for i, (x, y) in enumerate(positions[:len(ff)]):
    dx = 5 - x
    dy = 5 - y
    ax.annotate("", xy=(5 - 0.3 * np.sign(dx), 5 - 0.3 * np.sign(dy)),
                xytext=(x + 0.5 * np.sign(dx), y + 0.3 * np.sign(dy)),
                arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))

# Labels for semifinal games
ax.text(1.5, 5, "Semifinal 1", fontsize=9, ha="center", va="center", color="gray", style="italic")
ax.text(8.5, 5, "Semifinal 2", fontsize=9, ha="center", va="center", color="gray", style="italic")

plt.tight_layout()
plt.show()

## 6. Full Bracket

In [ ]:
print_bracket(prediction)

## 7. Region-by-Region Advancement Heatmap

In [ ]:
region_labels_map = {"W": "West", "X": "East", "Y": "South", "Z": "Midwest"}
round_names = {1: "R64", 2: "R32", 3: "S16", 4: "E8", 5: "F4", 6: "Champ"}

# Build team -> region mapping from bracket.teams
team_region = {}  # team_id -> region letter
team_seed_num = {}  # team_id -> seed number
team_name_map = {}  # team_id -> name
for seed_str, (tid, tname, snum) in bracket.teams.items():
    region = seed_str[0]
    if tid not in team_region or snum < team_seed_num.get(tid, 99):
        team_region[tid] = region
        team_seed_num[tid] = snum
        team_name_map[tid] = tname

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
regions = ["W", "X", "Y", "Z"]

for idx, region in enumerate(regions):
    ax = axes[idx // 2][idx % 2]
    
    # Get teams in this region, sorted by seed
    region_teams = [
        (tid, team_seed_num[tid], team_name_map[tid])
        for tid in team_region
        if team_region[tid] == region
    ]
    region_teams.sort(key=lambda x: x[1])
    
    # Build heatmap matrix: rows=teams, cols=rounds
    rounds_to_show = [1, 2, 3, 4, 5, 6]
    data = []
    labels = []
    for tid, seed, name in region_teams:
        adv = sim.advancement_probs.get(tid, {})
        row = [adv.get(r, 0.0) for r in rounds_to_show]
        data.append(row)
        labels.append(f"({seed}) {name}")
    
    if not data:
        ax.set_title(f"{region_labels_map.get(region, region)} — No data")
        continue
    
    heatmap_df = pd.DataFrame(
        data,
        index=labels,
        columns=[round_names[r] for r in rounds_to_show],
    )
    
    sns.heatmap(
        heatmap_df, ax=ax, annot=True, fmt=".2f", cmap="YlOrRd",
        vmin=0, vmax=1, linewidths=0.5, cbar_kws={"label": "Probability"},
    )
    ax.set_title(f"{region_labels_map.get(region, region)} Region", fontsize=13, fontweight="bold")
    ax.set_ylabel("")

plt.suptitle(f"{prediction.season} Advancement Probabilities by Region", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 8. Advancement Table

In [ ]:
print_advancement_table(prediction)

# Build styled DataFrame with background gradient
rows = []
for team_id, adv in sim.advancement_probs.items():
    name, seed, region = team_info.get(team_id, (f"Team{team_id}", 0, "?"))
    row = {"Team": f"({seed}) {name}", "Seed": seed, "Region": region}
    for rnd in [1, 2, 3, 4, 5, 6]:
        row[round_names[rnd]] = adv.get(rnd, 0.0)
    rows.append(row)

adv_df = pd.DataFrame(rows)
adv_df = adv_df.sort_values(["Champ", "F4", "E8"], ascending=False).head(32).reset_index(drop=True)

prob_cols = [round_names[r] for r in [1, 2, 3, 4, 5, 6]]
styled = (
    adv_df.style
    .background_gradient(subset=prob_cols, cmap="YlOrRd", vmin=0, vmax=1)
    .format({col: "{:.1%}" for col in prob_cols})
    .set_caption(f"{prediction.season} Top 32 Teams — Advancement Probabilities")
)
styled

## 9. R1 Expert Decomposition

In [ ]:
print_round1_decomposition(prediction)

r1_df = prediction.round1_decomposed
if r1_df is not None and not r1_df.empty:
    # Create matchup labels
    r1_df = r1_df.copy()
    r1_df["matchup"] = (
        "(" + r1_df["seed_a"].astype(int).astype(str) + ") " + r1_df["team_a"]
        + " v (" + r1_df["seed_b"].astype(int).astype(str) + ") " + r1_df["team_b"]
    )
    r1_df = r1_df.set_index("matchup")

    # 1x3 heatmap: expert predictions, gating weights, blended
    fig, axes = plt.subplots(1, 3, figsize=(20, max(8, len(r1_df) * 0.35)))

    # Panel 1: Expert predictions
    pred_cols = ["p_seed", "p_eff", "p_unc"]
    sns.heatmap(
        r1_df[pred_cols], ax=axes[0], annot=True, fmt=".3f",
        cmap="RdYlGn", center=0.5, vmin=0.3, vmax=0.95,
        linewidths=0.5, cbar_kws={"label": "P(higher seed wins)"},
    )
    axes[0].set_title("Expert Predictions", fontsize=12, fontweight="bold")
    axes[0].set_ylabel("")
    axes[0].set_xticklabels(["Seed", "Efficiency", "Uncertainty"], rotation=0)

    # Panel 2: Gating weights
    weight_cols = ["w_seed", "w_eff", "w_unc"]
    sns.heatmap(
        r1_df[weight_cols], ax=axes[1], annot=True, fmt=".3f",
        cmap="Blues", vmin=0, vmax=0.6,
        linewidths=0.5, cbar_kws={"label": "Gating Weight"},
    )
    axes[1].set_title("Gating Weights", fontsize=12, fontweight="bold")
    axes[1].set_ylabel("")
    axes[1].set_yticklabels([])
    axes[1].set_xticklabels(["Seed", "Efficiency", "Uncertainty"], rotation=0)

    # Panel 3: Blended probability
    sns.heatmap(
        r1_df[["p_blend"]], ax=axes[2], annot=True, fmt=".3f",
        cmap="RdYlGn", center=0.5, vmin=0.3, vmax=0.95,
        linewidths=0.5, cbar_kws={"label": "Blended P"},
    )
    axes[2].set_title("Blended Probability", fontsize=12, fontweight="bold")
    axes[2].set_ylabel("")
    axes[2].set_yticklabels([])
    axes[2].set_xticklabels(["Blend"], rotation=0)

    plt.suptitle(f"{prediction.season} Round 1 — Expert Decomposition", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No R1 decomposition data available.")

## 10. Upset Candidates

In [ ]:
if r1_df is not None and not r1_df.empty:
    # Identify upset candidates: games where the blended probability < 0.6
    # (i.e., MOE is less confident in the higher seed winning)
    upsets = r1_df[r1_df["p_blend"] < 0.6].copy()
    upsets["p_upset"] = 1 - upsets["p_blend"]
    upsets = upsets.sort_values("p_upset", ascending=False)

    if upsets.empty:
        print("No upset candidates found (all R1 blended probs >= 0.6).")
    else:
        print(f"Found {len(upsets)} upset candidates (p_blend < 0.6):")

        # Calculate each expert's contribution to the upset probability
        # Contribution = weight * (1 - expert_prediction)
        upsets["contrib_seed"] = upsets["w_seed"] * (1 - upsets["p_seed"])
        upsets["contrib_eff"] = upsets["w_eff"] * (1 - upsets["p_eff"])
        upsets["contrib_unc"] = upsets["w_unc"] * (1 - upsets["p_unc"])

        fig, ax = plt.subplots(figsize=(12, max(4, len(upsets) * 0.6)))

        y_pos = range(len(upsets))
        bar_height = 0.6

        # Stacked bars showing expert contribution to upset probability
        ax.barh(y_pos, upsets["contrib_seed"].values, height=bar_height,
                label="Seed Expert", color="#E24A33", edgecolor="black", linewidth=0.5)
        ax.barh(y_pos, upsets["contrib_eff"].values, height=bar_height,
                left=upsets["contrib_seed"].values,
                label="Efficiency Expert", color="#348ABD", edgecolor="black", linewidth=0.5)
        ax.barh(y_pos, upsets["contrib_unc"].values, height=bar_height,
                left=(upsets["contrib_seed"] + upsets["contrib_eff"]).values,
                label="Uncertainty Expert", color="#988ED5", edgecolor="black", linewidth=0.5)

        ax.set_yticks(y_pos)
        ax.set_yticklabels(upsets.index)
        ax.set_xlabel("Upset Probability (expert contribution)")
        ax.set_title(f"{prediction.season} Round 1 — Upset Candidates (p_blend < 0.6)")
        ax.legend(loc="lower right")

        # Add total upset probability as text
        for i, (idx, row) in enumerate(upsets.iterrows()):
            ax.text(row["p_upset"] + 0.005, i, f"{row['p_upset']:.1%}",
                    va="center", fontsize=9, fontweight="bold")

        plt.tight_layout()
        plt.show()
else:
    print("No R1 decomposition data available for upset analysis.")

## 11. MOE vs Log5 Bracket Comparison

In [ ]:
# Run Log5 simulation for comparison
bracket_2025 = load_bracket(2025)
log5_fn = build_log5_win_prob_fn(2025)
log5_sim = MCSimulator(50000, random_seed=42).simulate(bracket_2025, log5_fn)

# Build a TeamStatsLookup for name resolution
kenpom_df = load_kenpom()
train_df = merged_df[merged_df["season"] != 2025].copy()
train_deltas = compute_deltas(train_df)
upset_rate_lookup = build_upset_rate_lookup(train_deltas)
lookup = TeamStatsLookup(
    season=2025,
    kenpom_df=kenpom_df,
    upset_rate_lookup=upset_rate_lookup,
)

# Compare championship probs
moe_probs = prediction.simulation.championship_probs
log5_probs = log5_sim.championship_probs

# Merge into comparison table
all_team_ids = set(moe_probs.keys()) | set(log5_probs.keys())
champ_rows = []
for tid in all_team_ids:
    name = lookup.team_names.get(tid, f"Team{tid}")
    seed_info = lookup.team_seeds.get(tid, ("?", 0))
    champ_rows.append({
        "Team": f"({seed_info[1]}) {name}",
        "Seed": seed_info[1],
        "MOE_champ_%": moe_probs.get(tid, 0) * 100,
        "Log5_champ_%": log5_probs.get(tid, 0) * 100,
        "Diff_%": (moe_probs.get(tid, 0) - log5_probs.get(tid, 0)) * 100,
    })

champ_comp = pd.DataFrame(champ_rows).sort_values("MOE_champ_%", ascending=False).head(16)
print("Championship Probability Comparison (Top 16 by MOE):")
print(champ_comp.to_string(index=False, float_format="%.2f"))

# Compare EV bracket picks — find disagreements
moe_ev = {slot: tid for slot, tid in prediction.simulation.ev_bracket}
log5_ev = {slot: tid for slot, tid in log5_sim.ev_bracket}

disagreements = []
for slot in moe_ev:
    if moe_ev[slot] != log5_ev.get(slot):
        moe_name = lookup.team_names.get(moe_ev[slot], str(moe_ev[slot]))
        log5_name = lookup.team_names.get(log5_ev.get(slot, 0), str(log5_ev.get(slot, "?")))
        # Determine round from slot name
        if slot.startswith("R") and len(slot) > 1 and slot[1].isdigit():
            rnd = int(slot[1])
        else:
            rnd = 0
        disagreements.append({
            "Slot": slot,
            "Round": round_names.get(rnd, f"R{rnd}"),
            "MOE Pick": moe_name,
            "Log5 Pick": log5_name,
        })

disagree_df = pd.DataFrame(disagreements)
print(f"\nTotal EV bracket slots: {len(moe_ev)}")
print(f"Disagreements: {len(disagreements)} ({len(disagreements)/max(len(moe_ev),1)*100:.1f}%)")
print()
if not disagree_df.empty:
    print(disagree_df.to_string(index=False))
else:
    print("MOE and Log5 agree on all picks (unlikely but possible).")